[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AImotion-Bavaria/kipro_wi/blob/main/lecture2/harbour_ortools.ipynb)

# Ship Assignment in a Harbour with OR-Tools

Five boats have to be assigned to three docks.
Boats 1 and 2 are too large for dock 1.

In [ ]:
!pip install -q ortools

## Setup

The drawing code lives in `harbour_viz.py`. On Colab it is downloaded from the
GitHub repository (locally it is found next to the notebook).

When running locally, `autoreload` re-imports the module automatically, so changes to
`harbour_viz.py` take effect without restarting the kernel. On Colab it is switched off.

In [ ]:
import os
import urllib.request

if not os.path.exists("harbour_viz.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/AImotion-Bavaria/kipro_wi/main/lecture2/harbour_viz.py",
        "harbour_viz.py")

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

# autoreload only for local development (not needed on Colab)
if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

import matplotlib.pyplot as plt
import harbour_viz as viz

## Model

So far the model only has variables: each boat gets a dock.
The restriction for boats 1 and 2 is expressed by their smaller domain `2..3`.

In [ ]:
from ortools.sat.python import cp_model

# domains
Docks = range(1, 4)

model = cp_model.CpModel()

# variables
# boats 1 and 2 cannot use dock 1 -> smaller domain
boat1 = model.new_int_var(2, 3, "boat1")
boat2 = model.new_int_var(2, 3, "boat2")
boat3 = model.new_int_var(min(Docks), max(Docks), "boat3")
boat4 = model.new_int_var(min(Docks), max(Docks), "boat4")
boat5 = model.new_int_var(min(Docks), max(Docks), "boat5")

boats = {1: boat1, 2: boat2, 3: boat3, 4: boat4, 5: boat5}

## Solve

In [ ]:
solver = cp_model.CpSolver()
status = solver.solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    solution = {b: solver.value(var) for b, var in boats.items()}
    for b, dock in solution.items():
        print(f"boat{b} = {dock};")
else:
    print("=====UNSATISFIABLE=====")

## Visualisation

Each boat is drawn in the dock it is assigned to.
The drawing functions come from `harbour_viz.py`.

In [ ]:
viz.draw_harbour(solution)
plt.show()

## All solutions

Without any constraints, every combination of allowed docks is a solution:
$2 \cdot 2 \cdot 3 \cdot 3 \cdot 3 = 108$.
A solution callback collects them all.

In [ ]:
class CollectSolutions(cp_model.CpSolverSolutionCallback):
    def __init__(self, variables):
        super().__init__()
        self.variables = variables
        self.solutions = []

    def on_solution_callback(self):
        self.solutions.append({b: self.value(v) for b, v in self.variables.items()})


collector = CollectSolutions(boats)
solver = cp_model.CpSolver()
solver.parameters.enumerate_all_solutions = True
solver.solve(model, collector)

print(f"{len(collector.solutions)} solutions found")

Far too many to draw them all, so we show a random selection:

In [ ]:
import random

random.seed(1)
viz.draw_all(random.sample(collector.solutions, 6))
plt.show()